In [1]:
!pip install sentence-transformers numpy

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB

#### Imports

In [2]:
import boto3
import json
import numpy as np
from pathlib import Path
import sys
import tempfile

sys.path.append("../src")

from embeddings import load_embedding_model, create_embeddings

#### S3 paths

In [3]:
BUCKET_NAME = "YOUR_BUCKET_NAME"

CHUNKS_KEY = "chunks/paper_chunks.jsonl"

EMBEDDINGS_PREFIX = "embeddings/"
EMBEDDINGS_KEY = EMBEDDINGS_PREFIX + "chunk_embeddings.npy"
METADATA_KEY = EMBEDDINGS_PREFIX + "chunk_metadata.jsonl"

s3 = boto3.client("s3")

#### Load chunks from S3

In [4]:
obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=CHUNKS_KEY
)

jsonl_text = obj["Body"].read().decode("utf-8")

chunks = [
    json.loads(line)
    for line in jsonl_text.splitlines()
    if line.strip()
]

print(f"Loaded {len(chunks)} chunks.")

Loaded 660 chunks.


#### Prepare chunk texts and metadata

In [5]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_metadata = [
    {
        "source_file": chunk["source_file"],
        "paper_name": chunk["paper_name"],
        "chunk_id": chunk["chunk_id"],
        "start_word": chunk["start_word"],
        "end_word": chunk["end_word"],
        "text": chunk["text"],
    }
    for chunk in chunks
]

print(f"Prepared {len(chunk_texts)} texts for embedding.")

Prepared 660 texts for embedding.


#### Create Embeddings

In [7]:
model = load_embedding_model(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
embeddings = create_embeddings(
    texts=chunk_texts,
    model=model,
    batch_size=32,
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Embedding shape: (660, 384)


#### Save embeddings and metadata to S3

In [9]:
with tempfile.TemporaryDirectory() as tmpdir:
    embeddings_path = Path(tmpdir) / "chunk_embeddings.npy"
    metadata_path = Path(tmpdir) / "chunk_metadata.jsonl"

    np.save(embeddings_path, embeddings)

    metadata_jsonl = "\n".join(
        json.dumps(item, ensure_ascii=False)
        for item in chunk_metadata
    )

    metadata_path.write_text(metadata_jsonl, encoding="utf-8")

    s3.upload_file(
        str(embeddings_path),
        BUCKET_NAME,
        EMBEDDINGS_KEY,
    )

    s3.upload_file(
        str(metadata_path),
        BUCKET_NAME,
        METADATA_KEY,
    )

print(f"Saved embeddings to: s3://{BUCKET_NAME}/{EMBEDDINGS_KEY}")
print(f"Saved metadata to: s3://{BUCKET_NAME}/{METADATA_KEY}")

Saved embeddings to: s3://multiomic-vae-literature-rag-123223178042-eu-north-1-an/embeddings/chunk_embeddings.npy
Saved metadata to: s3://multiomic-vae-literature-rag-123223178042-eu-north-1-an/embeddings/chunk_metadata.jsonl


#### Verify S3 output

In [ ]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=EMBEDDINGS_PREFIX
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])